In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_entity_relationships_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "EntityRelationshipId    string,"
        #     "SourceSystem  string,"
        #     "PortfolioKey  string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp   timestamp,"
        #     "BatchId  string,"
        #     "EntityRelationshipKey   string,"
        #     "EntityKey    string,"
        #     "RelatedEntityKey string,"
        #     "EntityRelationshipType    string,"
        #     "EntityRelationshipSubType    string,"
        #     "EntityType    string,"
        #     "RelatedEntityType    string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.EntityRelationships"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== USER_DEFINED_FIELDS ==================

        USER_DEFINED_FIELDS_extra_filters = None

        if USER_DEFINED_FIELDS_extra_filters is not None:
            USER_DEFINED_FIELDS_extra_filters.append(date_filter)
            USER_DEFINED_FIELDS_final_filters = " AND ".join(
                USER_DEFINED_FIELDS_extra_filters
            )
        else:
            USER_DEFINED_FIELDS_final_filters = date_filter

        USER_DEFINED_FIELDS_selected_fields = [
            "FIELD_VALUE",
            "UD_ACCOUNT_NUMBER",
            "FIELD_LABEL",
            "PFG_TAG_SET_CODE",
            "businessdate",
        ]

        USER_DEFINED_FIELDS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_USER_DEFINED_FIELDS",
            USER_DEFINED_FIELDS_final_filters,
            USER_DEFINED_FIELDS_selected_fields,
        )

        # ================== loan_details ==================

        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = [
            "ACCOUNT_NUMBER",
            "ACC_STATUS",
            "ACC_OPEN_DT",
            "AGREEMENT_NUMBER",
            "prft_product_code",
            "prft_acc_unit",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        # ======================================================================

        # =================== M1_df_1 ==================
        M1_df_1 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"] == "2")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_1 ==================
        M2_df_1 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"].isin("2", "3", "4", "5", "6", "7"))
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .withColumnRenamed("FIELD_VALUE", "RELAT")
                .select("SLAVE_ACC", "RELAT")
            )

        # ================== Join Conditions ==================
        USER_DEFINED_FIELDS_M1_df_1_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(M1_df_1["MASTER_ACC"])
        )

        USER_DEFINED_FIELDS_M2_df_1_join_condition = (
            trim(USER_DEFINED_FIELDS["FIELD_VALUE"])
            == trim(M2_df_1["SLAVE_ACC"])
        )

        USER_DEFINED_FIELDS_loan_details_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(loan_details["ACCOUNT_NUMBER"])
        )

        loan_details_portfolio_join_condition = (
            loan_details["prft_acc_unit"] 
            == portfolio["unit_code_profits"]
        )

        # ================== Joins ==================

        result_df_1 = (
            USER_DEFINED_FIELDS.join(
                M1_df_1,
                USER_DEFINED_FIELDS_M1_df_1_join_condition,
                "INNER",
            )
            .join(
                M2_df_1,
                USER_DEFINED_FIELDS_M2_df_1_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                USER_DEFINED_FIELDS_loan_details_join_condition,
                "INNER",
            )
            .join(
                portfolio,
                loan_details_portfolio_join_condition,
                "left",
            )
        )

        result_df_1 = result_df_1.filter(
            (USER_DEFINED_FIELDS.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (USER_DEFINED_FIELDS.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (USER_DEFINED_FIELDS.FIELD_VALUE != '')
        )

        # HashKeys
        result_df_1_primary_key_fields = [
            USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
            USER_DEFINED_FIELDS.FIELD_VALUE,
            lit("Master - Slave Rltnp"),
            lit("profits"),
        ]

        result_df_1 = result_df_1.withColumn(
            "EntityRelationshipId",
            sha2(concat_ws("", *result_df_1_primary_key_fields), 256),
        ).withColumn(
            "EntityRelationshipKey",
            concat_ws(
                ";",
                USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
                USER_DEFINED_FIELDS.FIELD_VALUE,
                lit("Master - Slave Rltnp"),
            ),
        )

        # ================== Selection of Final Columns ==================
        result_df_1 = result_df_1.select(
            result_df_1.EntityRelationshipId.alias("EntityRelationshipId"),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df_1.EntityRelationshipKey.alias("EntityRelationshipKey"),
            trim(USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER).alias("EntityKey"),
            trim(USER_DEFINED_FIELDS.FIELD_VALUE).alias("RelatedEntityKey"),
            lit("Master - Slave Rltnp").alias("EntityRelationshipType"),
            when(M2_df_1.RELAT == "2", "ΜΛΤ")
            .when(M2_df_1.RELAT == "3", "ΤΟΚΟΙ Ν.3869")
            .when(M2_df_1.RELAT == "4", "WDN")
            .when(M2_df_1.RELAT == "5", "EXPENSES")
            .when(M2_df_1.RELAT == "6", "ADDIT. CR. CARD")
            .when(M2_df_1.RELAT == "7", "ΚΑΤΑΠΤΩΣΗ ΕΓΓΥΗΣΗ ΦΟΡΕΑ")
            .otherwise("ΑΓΝΩΣΤΟ")
            .alias("EntityRelationshipSubType"),
            lit("Exposure").alias("EntityType"),
            lit("Exposure").alias("RelatedEntityType"),
            USER_DEFINED_FIELDS.businessdate.alias("BusinessDate")
        )
        # ======================================================================

        # =================== M1_df_2 ==================
        M1_df_2 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"] == "6")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_2 ==================
        M2_df_2 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"] == "8")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .withColumnRenamed("FIELD_VALUE", "RELAT")
                .select("SLAVE_ACC", "RELAT")
            )

        # ================== Join Conditions ==================
        USER_DEFINED_FIELDS_M1_df_2_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(M1_df_2["MASTER_ACC"])
        )

        USER_DEFINED_FIELDS_M2_df_2_join_condition = (
            trim(USER_DEFINED_FIELDS["FIELD_VALUE"])
            == trim(M2_df_2["SLAVE_ACC"])
        )

        USER_DEFINED_FIELDS_loan_details_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(loan_details["ACCOUNT_NUMBER"])
        )

        loan_details_portfolio_join_condition = (
            loan_details["prft_acc_unit"] 
            == portfolio["unit_code_profits"]
        )

        # ================== Joins ==================

        result_df_2 = (
            USER_DEFINED_FIELDS.join(
                M1_df_2,
                USER_DEFINED_FIELDS_M1_df_2_join_condition,
                "INNER",
            )
            .join(
                M2_df_2,
                USER_DEFINED_FIELDS_M2_df_2_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                USER_DEFINED_FIELDS_loan_details_join_condition,
                "INNER",
            )
            .join(
                portfolio,
                loan_details_portfolio_join_condition,
                "left",
            )
        )

        result_df_2 = result_df_2.filter(
            (USER_DEFINED_FIELDS.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (USER_DEFINED_FIELDS.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (USER_DEFINED_FIELDS.FIELD_VALUE != '')
        )

        # HashKeys
        result_df_2_primary_key_fields = [
            USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
            USER_DEFINED_FIELDS.FIELD_VALUE,
            lit("Master - Slave Rltnp"),
            lit("profits"),
        ]

        result_df_2 = result_df_2.withColumn(
            "EntityRelationshipId",
            sha2(concat_ws("", *result_df_2_primary_key_fields), 256),
        ).withColumn(
            "EntityRelationshipKey",
            concat_ws(
                ";",
                USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
                USER_DEFINED_FIELDS.FIELD_VALUE,
                lit("Master - Slave Rltnp"),
            ),
        )

        # ================== Selection of Final Columns ==================
        result_df_2 = result_df_2.select(
            result_df_2.EntityRelationshipId.alias("EntityRelationshipId"),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df_2.EntityRelationshipKey.alias("EntityRelationshipKey"),
            trim(USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER).alias("EntityKey"),
            trim(USER_DEFINED_FIELDS.FIELD_VALUE).alias("RelatedEntityKey"),
            lit("Master - Slave Rltnp").alias("EntityRelationshipType"),
            when(M2_df_2.RELAT == "8", "EXT").otherwise("ΑΓΝΩΣΤΟ").alias("EntityRelationshipSubType"),
            lit("Exposure").alias("EntityType"),
            lit("Exposure").alias("RelatedEntityType"),
            USER_DEFINED_FIELDS.businessdate.alias("BusinessDate")
        )
        # ======================================================================

        # =================== M1_df_3 ==================
        M1_df_3 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"] == "3")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_3 ==================
        M2_df_3 = (USER_DEFINED_FIELDS.where(
                    (USER_DEFINED_FIELDS["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (USER_DEFINED_FIELDS["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (USER_DEFINED_FIELDS["FIELD_VALUE"] == "4")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .withColumnRenamed("FIELD_VALUE", "RELAT")
                .select("SLAVE_ACC", "RELAT")
            )

        # ================== Join Conditions ==================
        USER_DEFINED_FIELDS_M1_df_3_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(M1_df_3["MASTER_ACC"])
        )

        USER_DEFINED_FIELDS_M2_df_3_join_condition = (
            trim(USER_DEFINED_FIELDS["FIELD_VALUE"])
            == trim(M2_df_3["SLAVE_ACC"])
        )

        USER_DEFINED_FIELDS_loan_details_join_condition = (
            trim(USER_DEFINED_FIELDS["UD_ACCOUNT_NUMBER"])
            == trim(loan_details["ACCOUNT_NUMBER"])
        )

        loan_details_portfolio_join_condition = (
            loan_details["prft_acc_unit"] 
            == portfolio["unit_code_profits"]
        )

        # ================== Joins ==================

        result_df_3 = (
            USER_DEFINED_FIELDS.join(
                M1_df_3,
                USER_DEFINED_FIELDS_M1_df_3_join_condition,
                "INNER",
            )
            .join(
                M2_df_3,
                USER_DEFINED_FIELDS_M2_df_3_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                USER_DEFINED_FIELDS_loan_details_join_condition,
                "INNER",
            )
            .join(
                portfolio,
                loan_details_portfolio_join_condition,
                "left",
            )
        )

        result_df_3 = result_df_3.filter(
            (USER_DEFINED_FIELDS.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (USER_DEFINED_FIELDS.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (USER_DEFINED_FIELDS.FIELD_VALUE != '')
        )

        # HashKeys
        result_df_3_primary_key_fields = [
            USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
            USER_DEFINED_FIELDS.FIELD_VALUE,
            lit("Master - Slave Rltnp"),
            lit("profits"),
        ]

        result_df_3 = result_df_3.withColumn(
            "EntityRelationshipId",
            sha2(concat_ws("", *result_df_3_primary_key_fields), 256),
        ).withColumn(
            "EntityRelationshipKey",
            concat_ws(
                ";",
                USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER,
                USER_DEFINED_FIELDS.FIELD_VALUE,
            lit("Master - Slave Rltnp"),
            ),
        )

        # ================== Selection of Final Columns ==================
        result_df_3 = result_df_3.select(
            result_df_3.EntityRelationshipId.alias("EntityRelationshipId"),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df_3.EntityRelationshipKey.alias("EntityRelationshipKey"),
            trim(USER_DEFINED_FIELDS.UD_ACCOUNT_NUMBER).alias("EntityKey"),
            trim(USER_DEFINED_FIELDS.FIELD_VALUE).alias("RelatedEntityKey"),
            lit("Master - Slave Rltnp").alias("EntityRelationshipType"),
            when(M2_df_3.RELAT == "4", "WDN").otherwise("ΑΓΝΩΣΤΟ").alias("EntityRelationshipSubType"),
            lit("Exposure").alias("EntityType"),
            lit("Exposure").alias("RelatedEntityType"),
            USER_DEFINED_FIELDS.businessdate.alias("BusinessDate")
        )

        # ================== union all ==================
        result_df_profits = (
            result_df_1.union(result_df_2)
                       .union(result_df_3)
        )

        result_df_profits = result_df_profits.filter(
            (result_df_profits.EntityKey != result_df_profits.RelatedEntityKey)
        )

        # ================== Append ==================
        result_df_profits = (
            result_df_profits.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_entity_relationships_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)